# Word2Vec — Custom Training on a Text Corpus

**Word2Vec** is a neural network technique that learns **dense vector representations (embeddings)** for words. Unlike sparse representations (OHE, BoW, TF-IDF), Word2Vec produces compact, meaningful vectors where **semantically similar words are close together** in vector space.

### The Core Idea — Distributional Hypothesis
> *"A word is known by the company it keeps."*  
> Words that appear in similar contexts will have similar vectors.

### Two Training Architectures

| Architecture | Task | Best for |
|---|---|---|
| **CBOW** (`sg=0`, default) | Predict center word from context | Frequent words, faster training |
| **Skip-Gram** (`sg=1`) | Predict context words from center | Rare words, larger datasets |

### Key Hyperparameters

| Parameter | Meaning |
|---|---|
| `vector_size` | Dimensionality of each word embedding |
| `window` | Number of words to look at on each side of a target word |
| `min_count` | Ignore words appearing fewer than this many times |
| `epochs` | Number of full passes over the training corpus |

### Pipeline Overview
```
Raw Text Files
      ↓
Sentence Tokenization (NLTK)
      ↓
Word Tokenization (Gensim simple_preprocess)
      ↓
Build Vocabulary
      ↓
Train Word2Vec Model
      ↓
Query Embeddings & Similarities
```

## Step 1: Import Libraries

- **`pandas` / `numpy`** — standard data manipulation libraries.
- **`os`** — used to list and navigate files in the data directory.
- **`gensim`** — the NLP library that provides the `Word2Vec` implementation.
- **`nltk`** — Natural Language Toolkit, used here for sentence tokenization.

In [1]:
import pandas as pd
import numpy as np
import os
import gensim
import nltk

## Step 2: Download NLTK Resources

`nltk.download('all')` downloads all NLTK datasets and models, including `punkt` — the pre-trained sentence tokenizer we use in the next step.

You only need to run this once. On subsequent runs you can comment it out.

In [2]:
nltk.download('all')

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to
[nltk_data]    |     C:\Users\prbhatia\AppData\Roaming\nltk_data...
[nltk_data]    |   Package abc is already up-to-date!
[nltk_data]    | Downloading package alpino to
[nltk_data]    |     C:\Users\prbhatia\AppData\Roaming\nltk_data...
[nltk_data]    |   Package alpino is already up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     C:\Users\prbhatia\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger is already up-
[nltk_data]    |       to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     C:\Users\prbhatia\AppData\Roaming\nltk_data...
[nltk_data]    |   Package averaged_perceptron_tagger_eng is already
[nltk_data]    |       up-to-date!
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     C:\Users\prbhatia\AppData\Roaming

True

## Step 3: Import Tokenization Helpers

- **`sent_tokenize`** — splits a block of text into individual **sentences** using punctuation and language rules.
- **`simple_preprocess`** — Gensim utility that lowercases text, removes punctuation, and splits it into a **list of word tokens**. It is fast and well-suited for Word2Vec input.

In [3]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

## Step 4: Inspect the Data Folder

List all files in the `./data/` directory to confirm which text files are available for training.

Word2Vec learns better with **more data** — the more diverse and larger the corpus, the richer the embeddings.

In [4]:
for file in os.listdir('./data/'):
    print(file)

story1.txt
story2.txt


## Step 5: Build the Training Corpus

This is the **most important preprocessing step**. For every file in the data folder:

1. **Open the file** with `unicode_escape` encoding to handle special characters.
2. **Read the full text** into a string (`corpus`).
3. **Sentence tokenize** — split the text into sentences using `sent_tokenize`.
4. **Word tokenize** — convert each sentence into a list of lowercase word tokens using `simple_preprocess`.
5. **Append** each tokenized sentence to the `story` list.

The final `story` variable is a **list of lists** — each inner list is one tokenized sentence:
```python
[
    ['arjun', 'was', 'a', 'farmer'],
    ['he', 'worked', 'hard', 'every', 'day'],
    ...
]
```
This is the exact format `Word2Vec` expects as training input.

In [5]:
folder_path = r"C:\Training\Modern Route-Full Stack GenerativeAI And Agentic AI\Full-Stack-GenAI-Bootcamp-1.0\Class-06-07-11-12-Apr-Word2vec-with-practical\data"
story = []
for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    with open(file_path, encoding='unicode_escape') as f:
        corpus = f.read()
    raw_sent = sent_tokenize(corpus)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

print(f"Total tokenized sentences: {len(story)}")
print(f"Example - first sentence: {story[0]}")

Total tokenized sentences: 44
Example - first sentence: ['once', 'upon', 'time', 'there', 'was', 'young', 'boy', 'named', 'arjun', 'who', 'lived', 'in', 'small', 'town', 'surrounded', 'by', 'hills', 'and', 'rivers']


## Step 6: Inspect the Training Corpus

Before training, verify a few sentences to make sure preprocessing worked correctly.

All text should be:
- **Lowercase** — `Word2Vec` treats `Arjun` and `arjun` as the same token.
- **Punctuation-free** — `simple_preprocess` removes punctuation.
- **Split into word lists** — each sentence is a Python `list` of strings.

In [6]:
print("Total sentences:", len(story))
print("Sentence 1:", story[0])
print("Sentence 2:", story[1])

Total sentences: 44
Sentence 1: ['once', 'upon', 'time', 'there', 'was', 'young', 'boy', 'named', 'arjun', 'who', 'lived', 'in', 'small', 'town', 'surrounded', 'by', 'hills', 'and', 'rivers']
Sentence 2: ['he', 'was', 'very', 'curious', 'about', 'how', 'things', 'worked', 'and', 'always', 'asked', 'questions']


## Step 7: Initialize the Word2Vec Model

Create the model object and configure its **hyperparameters**:

| Hyperparameter | Value | Explanation |
|---|---|---|
| `window=10` | 10 | Look at 10 words to the left and right of each target word for context |
| `min_count=1` | 1 | Keep all words, even those appearing only once (use higher values for large corpora) |
| `vector_size=150` | 150 | Each word will be represented as a 150-dimensional vector |
| `sg` | not set → 0 | Uses **CBOW** by default; set `sg=1` for Skip-Gram |

> **Note:** This cell only **initializes** the model. No training happens here yet.

In [62]:
# sg=0 → CBOW (default)  |  sg=1 → Skip-Gram
custom_model = gensim.models.Word2Vec(
    window=150,
    min_count=1,
    vector_size=150
)

## Step 8: Build the Vocabulary

`build_vocab(story)` scans the entire training corpus and:
- Collects all **unique words** that meet the `min_count` threshold.
- Assigns each word an **internal index**.
- Initializes the **weight matrices** of the neural network.

This step must be done **before** calling `.train()`. Think of it as building a dictionary the model will use during training.

In [63]:
custom_model.build_vocab(story)
print("Vocabulary size:", len(custom_model.wv))
print("Corpus sentence count:", custom_model.corpus_count)

Vocabulary size: 257
Corpus sentence count: 44


## Step 9: Train the Model

`.train()` runs the actual **backpropagation** through the Word2Vec neural network:

- **`story`** — the tokenized training data.
- **`total_examples`** — total number of sentences; used internally to track training progress.
- **`epochs=10`** — the model makes **10 full passes** over the entire corpus, adjusting word vectors each time.

The return value `(X, Y)` shows:
- `X` — number of training examples processed.
- `Y` — total number of raw words processed across all epochs.

> **Tip:** More epochs + more data = better embeddings, but longer training time.

In [64]:
custom_model.train(story, total_examples=custom_model.corpus_count, epochs=10)
print("Training complete!")

Training complete!


## Step 10: Inspect Word Embeddings

`model.wv["word"]` returns the **learned embedding vector** for a word — a NumPy array of `vector_size` (150) floats.

These numbers are not directly interpretable on their own, but their **relative positions** in 150-dimensional space encode semantic meaning.

Words used in similar contexts (like `arjun` and `engineer` in tech stories) will have vectors that **point in similar directions**.

In [65]:
print("Vector for 'arjun':")
print(custom_model.wv["arjun"])
print("\nVector dimensions:", len(custom_model.wv["arjun"]))

Vector for 'arjun':
[ 2.4612860e-03  2.3782530e-03  6.1284294e-03 -3.1914732e-03
 -7.9795852e-04 -4.7829300e-03 -6.5579312e-03 -4.6264650e-03
 -1.3539006e-03 -3.9741951e-03  3.3819808e-03 -3.9095455e-03
  8.1187132e-04  3.4147609e-04 -2.7141725e-03 -1.0150652e-03
  6.8327230e-03  5.9329676e-03 -3.3021828e-03  6.6131195e-03
 -4.2011868e-03  3.8028995e-03 -1.1373975e-03  2.9808267e-03
  1.9193118e-03  4.0853801e-03 -2.4715366e-03  5.1790224e-03
  4.8473445e-03 -7.1768477e-03 -5.0934167e-03 -5.1303641e-03
  1.4341145e-03 -2.4457558e-03 -5.5016191e-03 -5.0878269e-03
  6.4890636e-03  1.7011459e-03 -6.2870234e-03 -3.8511609e-03
  2.2138937e-03 -3.1347028e-03  2.7609058e-03 -2.9973884e-03
  1.9054014e-03 -5.3333603e-03  4.2239600e-03  2.9903634e-03
  4.5821234e-04  2.3567781e-03 -5.8409316e-03  1.2543228e-03
 -9.5484237e-04 -5.9053889e-03 -6.4458605e-03 -5.5450085e-04
  2.7744123e-03  3.0330347e-03 -4.3509672e-03 -3.9945822e-03
 -3.7788581e-03 -2.6535818e-03 -4.7315275e-03 -6.7891167e-03
 -1.

In [66]:
print("Vector for 'farmers':")
print(custom_model.wv["farmers"])

Vector for 'farmers':
[ 5.2222181e-03  4.8161810e-03 -4.1939993e-03 -5.5491640e-03
  5.5146823e-03 -2.3239099e-03 -2.7697387e-03  6.1674058e-03
  3.0095237e-03  2.9345991e-03 -6.8432232e-04  5.9816148e-03
  4.7731875e-03  4.4768672e-03  5.5777207e-03 -4.3120454e-03
  8.6409209e-04 -1.3020883e-03  3.9727185e-03  7.1106837e-03
  2.9809955e-03  2.5487007e-03  5.3286543e-03  4.6113264e-03
 -5.2672704e-06 -6.0940417e-03 -2.3549970e-03  5.3364057e-03
 -3.5807982e-03 -1.2540020e-04  3.4608562e-03 -4.5855753e-03
  5.0152689e-03  3.0938245e-04  7.3102285e-04  5.7884445e-03
  1.8341603e-03  1.0403557e-03 -6.3493010e-03  5.9665642e-03
 -6.3921306e-03  5.7063829e-03  2.5973041e-03 -2.7112939e-04
 -5.2019530e-03  6.2623378e-03 -9.2615880e-04  3.3455594e-03
 -2.0593510e-03 -2.9373493e-03  6.7251225e-05 -5.4893936e-03
 -4.7991164e-03 -6.0266457e-03  3.4150479e-03  6.9484994e-04
 -8.2597916e-04  4.0611797e-03 -1.3896772e-03 -4.3690223e-03
  1.4168924e-03 -5.2374513e-03  4.3408686e-04  6.0107019e-03
 -

In [67]:
print("Vector for 'ai':")
print(custom_model.wv["ai"])

Vector for 'ai':
[ 6.4169192e-03  4.2191953e-03 -6.2914547e-03 -4.7623869e-03
 -5.5429065e-03  2.8501493e-03  6.0144267e-03 -3.1232741e-03
 -6.1115865e-03 -8.0672622e-04 -1.7931935e-03  9.3792670e-04
  3.3184437e-03  3.4307438e-04  3.0815725e-03  6.0941414e-03
 -5.1837582e-03  3.8401524e-03  2.8856812e-04 -5.2647996e-03
 -3.7334063e-03  2.7447082e-03  5.5045625e-03 -4.1215480e-03
 -2.9240309e-03 -1.9536051e-03 -2.5270344e-03 -6.5571088e-03
  9.8618376e-04 -5.2694427e-03 -4.0760245e-03  2.9243333e-03
 -3.8895111e-03 -2.1446503e-03  1.7331062e-04  6.2909559e-03
 -3.4178512e-03  7.0607581e-04 -4.6939943e-03  4.2261058e-03
  6.5917657e-03  6.5910709e-03 -2.9453500e-03  6.3574738e-03
  4.8035514e-03 -1.7702795e-04 -2.3599253e-03  9.9901156e-04
  2.4933105e-03 -4.7469563e-03  2.2688690e-03  1.9616652e-03
  4.2202312e-04 -1.8131216e-03  1.6159589e-03  3.8284846e-03
 -2.3534838e-03  1.0800229e-03  3.7473433e-03  8.2701963e-04
  5.1085618e-03  3.4572815e-03 -2.4175856e-03  4.4030966e-03
  1.673

## Step 11: Find Most Similar Words

`.most_similar("word")` returns the **top 10 nearest neighbors** in the embedding space using **cosine similarity**.

Cosine similarity measures the angle between two vectors:
- Score **close to 1.0** → words are very similar in context.
- Score **close to 0.0** → words are unrelated.

This is the key power of Word2Vec — words that never directly appeared together can still be recognized as semantically related.

In [68]:
print("Most similar to 'arjun':")
print(custom_model.wv.most_similar("arjun"))

Most similar to 'arjun':
[('that', 0.28096339106559753), ('curious', 0.2643823027610779), ('how', 0.23491014540195465), ('their', 0.2232692539691925), ('predictions', 0.21222464740276337), ('healthcare', 0.21134436130523682), ('passed', 0.19456259906291962), ('giving', 0.1766764521598816), ('as', 0.17515698075294495), ('improvements', 0.17153988778591156)]


In [69]:
print("Most similar to 'ai':")
print(custom_model.wv.most_similar("ai"))

Most similar to 'ai':
[('using', 0.20115767419338226), ('complex', 0.19935138523578644), ('many', 0.1978316754102707), ('inspired', 0.18830694258213043), ('better', 0.18560890853405), ('saw', 0.1780748963356018), ('happy', 0.17554093897342682), ('knowledge', 0.1729685217142105), ('hard', 0.17112012207508087), ('worked', 0.17050756514072418)]


In [70]:
print("Most similar to 'engineer':")
print(custom_model.wv.most_similar("engineer"))

Most similar to 'engineer':
[('solve', 0.21457235515117645), ('were', 0.20142193138599396), ('boy', 0.2012481391429901), ('decided', 0.20122012495994568), ('build', 0.19754533469676971), ('learned', 0.18548360466957092), ('refining', 0.1781998574733734), ('simple', 0.17227141559123993), ('shared', 0.17194941639900208), ('named', 0.1651814728975296)]


In [71]:
print("Most similar to 'data':")
print(custom_model.wv.most_similar("data"))

Most similar to 'data':
[('world', 0.23556368052959442), ('stuck', 0.22956547141075134), ('became', 0.19082430005073547), ('better', 0.1868472546339035), ('in', 0.18525658547878265), ('idea', 0.18410032987594604), ('villages', 0.1740354746580124), ('struggled', 0.165379136800766), ('think', 0.16523075103759766), ('things', 0.16465523838996887)]


## Step 12: Find the Odd Word Out

`.doesnt_match([list of words])` identifies the **word that is least similar** to the others in the group.

It works by computing the **mean vector** of all words in the list, then finding which word's vector is farthest from that mean.

This is a great test to verify that the model has learned **meaningful groupings** of concepts.

In [72]:
# 'farmers' is the odd one out among tech/coding words
print(custom_model.wv.doesnt_match(["arjun", "engineer", "programming", "farmers"]))

engineer


In [73]:
# 'river' is the odd one out among tech/data words
print(custom_model.wv.doesnt_match(["data", "model", "system", "river"]))

system


In [74]:
# 'technology' is the odd one out among farming words
print(custom_model.wv.doesnt_match(["weather", "crops", "farmers", "technology"]))

crops


## Step 13: Measure Word Similarity Score

`.similarity("word1", "word2")` returns a single **cosine similarity score** between two words.

- **Score near `1.0`** → words appear in very similar contexts → semantically related.
- **Score near `0.0`** → words appear in very different contexts → unrelated.
- **Negative score** → words appear in opposite contexts.

The quality of these scores depends heavily on the **size and diversity** of the training corpus.

In [75]:
score = custom_model.wv.similarity("ai", "technology")
print(f"Similarity between 'ai' and 'technology': {score:.4f}")

Similarity between 'ai' and 'technology': 0.0213


In [76]:
score = custom_model.wv.similarity("arjun", "engineer")
print(f"Similarity between 'arjun' and 'engineer': {score:.4f}")

Similarity between 'arjun' and 'engineer': -0.0545


## Summary

| Step | Action | Purpose |
|------|--------|---------|
| 1 | Import libraries | Load pandas, numpy, os, gensim, nltk |
| 2 | Download NLTK data | Get the sentence tokenizer and other NLTK resources |
| 3 | Import tokenizers | Load `sent_tokenize` and `simple_preprocess` |
| 4 | Inspect data folder | Check which training files are available |
| 5 | Build training corpus | Read all files → sentence split → word tokenize → list of lists |
| 6 | Inspect corpus | Verify tokenization output before training |
| 7 | Initialize model | Set hyperparameters: `window`, `min_count`, `vector_size` |
| 8 | Build vocabulary | Scan corpus, collect unique words, initialize weight matrices |
| 9 | Train model | Run backprop for N epochs to learn word embeddings |
| 10 | Inspect embeddings | View raw vector values for individual words |
| 11 | Find similar words | Query nearest neighbors using cosine similarity |
| 12 | Find odd word out | Test semantic groupings using mean vector distance |
| 13 | Measure similarity | Get a single similarity score between any two words |

## Assignment / Next Steps

1. **Add more data** — collect larger and more diverse text files into the `data/` folder.
2. **Train for more epochs** — try `epochs=50` or `epochs=100` and observe how similarity scores change.
3. **Experiment with `vector_size`** — try `50`, `100`, `300` and compare the quality of similarities.
4. **Try Skip-Gram** — set `sg=1` and compare results with the default CBOW mode.
5. **Visualize embeddings** — use `TSNE` or `PCA` to reduce vectors to 2D and plot them.

> **Limitation of Word2Vec:** Each word gets one **static vector** regardless of context. The word `bank` has the same embedding whether it means a river bank or a financial bank. This is solved by **contextual embeddings** like BERT and GPT.